# Viettel AI Race 2026 — Colab validation workflow

Colab T4 is used for startup, stability, workload, and accuracy checks only. Its latency must not be used to rank H200 FP8 submissions. Open this notebook from GitHub; the first setup cell clones or updates the repository and installs the CUDA 12.9 build of vLLM 0.22.1.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

REPO_URL = 'https://github.com/Platypus27-coder/viettel-ai-race-llm-serving.git'
REPO_BRANCH = 'main'
PROJECT_DIR = Path('/content/viettel-ai-race-llm-serving')

if (PROJECT_DIR / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH],
        check=True,
    )
elif PROJECT_DIR.exists():
    raise RuntimeError(f'{PROJECT_DIR} exists but is not a Git repository')
else:
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

subprocess.run(['nvidia-smi'], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'],
    check=True,
)
UV_BIN = shutil.which('uv')
if not UV_BIN:
    raise RuntimeError('uv was installed but is not available on PATH')

# A rerun must remove a previously installed CUDA 13 wheel first.
subprocess.run([
    UV_BIN, 'pip', 'uninstall', '--system',
    'vllm', 'torch', 'torchvision', 'torchaudio',
], check=False)
subprocess.run([
    UV_BIN, 'pip', 'install', '--system',
    '--torch-backend=cu129',
    '--extra-index-url', 'https://wheels.vllm.ai/0.22.1/cu129',
    '--index-strategy', 'unsafe-best-match',
    'vllm==0.22.1', 'lm-eval[api]>=0.4.9', 'aiohttp', 'openai',
    'numpy', 'transformers>=4.57.2', 'huggingface_hub',
], check=True)

assert (PROJECT_DIR / 'benchmark/benchmark_ers.py').is_file()
commit = subprocess.check_output(
    ['git', '-C', str(PROJECT_DIR), 'rev-parse', '--short', 'HEAD'],
    text=True,
).strip()

preflight_code = r'''
import json
from importlib.metadata import version
from packaging.version import Version
import torch
import vllm
import vllm._C

package_version = version('vllm')
assert Version(package_version).base_version == '0.22.1', package_version
assert 'cu129' in package_version, f'Expected cu129 wheel, got {package_version}'
assert torch.version.cuda and torch.version.cuda.startswith('12.'), torch.version.cuda
assert torch.cuda.is_available(), 'CUDA is not available to PyTorch'
gpu = torch.cuda.get_device_name(0)
assert 'T4' in gpu, f'Expected a Tesla T4 runtime, got {gpu}'
print(json.dumps({
    'vllm_version': vllm.__version__,
    'vllm_package_version': package_version,
    'torch_version': torch.__version__,
    'torch_cuda': torch.version.cuda,
    'gpu': gpu,
}))
'''
preflight = subprocess.run(
    [sys.executable, '-c', preflight_code],
    text=True,
    capture_output=True,
)
if preflight.returncode != 0:
    raise RuntimeError(
        'CUDA 12.9 vLLM preflight failed. Use Runtime > Restart session, '
        'then run this setup cell again.\n'
        f'STDOUT:\n{preflight.stdout}\nSTDERR:\n{preflight.stderr}'
    )
environment_info = json.loads(preflight.stdout.strip().splitlines()[-1])
environment_info.update({
    'repo_commit': commit,
    'repo_url': REPO_URL,
    'vllm_wheel_variant': 'cu129',
})
ENVIRONMENT_PATH = Path('/content/environment.json')
ENVIRONMENT_PATH.write_text(
    json.dumps(environment_info, indent=2),
    encoding='utf-8',
)
print(f'Repository ready at {PROJECT_DIR} ({commit})')
print(json.dumps(environment_info, indent=2))


In [ ]:
from huggingface_hub import snapshot_download

MODEL_DIR = Path('/content/model')
snapshot_download(
    repo_id='LiquidAI/LFM2.5-1.2B-Instruct',
    local_dir=MODEL_DIR,
)


## Start a clean server

Set `PRECISION` to `baseline` or `fp8`. Restart the server before every measured configuration. On T4, baseline uses FP16 and FP8 uses the non-Hopper compatibility path.

In [ ]:
import os, signal, subprocess, sys, time
import requests, torch
from pathlib import Path

PRECISION = 'baseline'  # baseline | fp8
BATCH_TOKENS = 4096     # run 3: 4096 | run 4: 2048
MAX_NUM_SEQS = 64       # FP8 winner: 64 | BF16 fallback: 48
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
if 'T4' in GPU_NAME:
    print('WARNING: T4 latency is not an H200 performance signal.')

old_pid = globals().get('server_process')
if old_pid is not None and old_pid.poll() is None:
    print('Terminating old server process...')
    old_pid.terminate()
    try:
        old_pid.wait(timeout=30)
    except subprocess.TimeoutExpired:
        print('Old server process did not terminate gracefully, sending SIGKILL...')
        old_pid.kill()
        old_pid.wait()  # Wait for it to be truly gone
        # Print the log if cleanup was necessary
        if Path('/content/vllm.log').exists():
            full_log = Path('/content/vllm.log').read_text(encoding='utf-8')
            print('vLLM server log during old server termination attempt:\n', full_log)

command = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    f'--model={MODEL_DIR}',
    '--served-model-name=LFM2.5-1.2B-Instruct',
    '--host=0.0.0.0', '--port=8000', '--tensor-parallel-size=1',
    '--dtype=float16', '--gpu-memory-utilization=0.90',
    '--max-model-len=8192', '--enable-prefix-caching',
    '--enable-chunked-prefill', f'--max-num-batched-tokens={BATCH_TOKENS}',
    f'--max-num-seqs={MAX_NUM_SEQS}',
]
if PRECISION == 'fp8':
    command.append('--quantization=fp8')

env = os.environ.copy()
env.update({'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1', 'VLLM_NO_USAGE_STATS': '1', 'DO_NOT_TRACK': '1'})
log_handle = open('/content/vllm.log', 'w', encoding='utf-8')
server_process = subprocess.Popen(command, stdout=log_handle, stderr=subprocess.STDOUT, env=env)
for attempt in range(120):
    try:
        if requests.get('http://localhost:8000/health', timeout=3).status_code == 200:
            print('Server ready; prefix cache is still cold.')
            break
    except requests.RequestException:
        pass
    if server_process.poll() is not None:
        log_tail = Path('/content/vllm.log').read_text(encoding='utf-8')[-5000:]
        hint = ''
        if 'libcudart.so.13' in log_tail:
            hint = ('CUDA 13 wheel detected. Restart the Colab session and '
                    'rerun the setup cell before starting the server.\n')
        raise RuntimeError(hint + log_tail)
    time.sleep(2)
else:
    # Ensure logs are flushed before reading
    log_handle.close()
    full_log = Path('/content/vllm.log').read_text(encoding='utf-8')
    print('vLLM server log:\n', full_log)
    raise TimeoutError('vLLM did not become healthy')


## Run the published 420-request trace

Start with one rate. Use the sweep only when runtime allows; each run begins with a newly restarted server so the shared prefix is not prewarmed.

In [ ]:
TRACE = PROJECT_DIR / '019e649f-4e27-74db-82da-920f57b13786/grading-workload-spec.json'
RESULT = Path(f'/content/ers-{PRECISION}-batch{BATCH_TOKENS}.json')
subprocess.run([
    sys.executable, str(PROJECT_DIR / 'benchmark/benchmark_ers.py'),
    '--trace', str(TRACE), '--tokenizer-path', str(MODEL_DIR),
    '--request-rate', 'inf', '--seed', '42', '--runs', '1',
    '--output', str(RESULT),
], check=True)


## Accuracy gates

Run the quick check first. Run full GPQA for baseline and every quantized candidate. Use `--task gpqa_diamond` to match the competition task name; if the installed harness lists a versioned Diamond alias, pass that exact alias instead.

In [ ]:
accuracy_script = PROJECT_DIR / 'benchmark/test_accuracy.py'
subprocess.run([sys.executable, str(accuracy_script), '--mode', 'quick'], check=True)

# Full gate (uncomment after the quick check succeeds):
# subprocess.run([
#     sys.executable, str(accuracy_script), '--mode', 'gpqa',
#     '--task', 'gpqa_diamond', '--concurrency', '4',
#     '--output', f'/content/gpqa-{PRECISION}-batch{BATCH_TOKENS}',
# ], check=True)


In [ ]:
import shutil
artifact_dir = Path('/content/run-artifacts')
artifact_dir.mkdir(exist_ok=True)
shutil.copy2(RESULT, artifact_dir / RESULT.name)
shutil.copy2('/content/vllm.log', artifact_dir / 'vllm.log')
shutil.copy2(ENVIRONMENT_PATH, artifact_dir / 'environment.json')
gpqa_dir = Path(f'/content/gpqa-{PRECISION}-batch{BATCH_TOKENS}')
if gpqa_dir.exists():
    shutil.copytree(gpqa_dir, artifact_dir / 'gpqa', dirs_exist_ok=True)
artifact = shutil.make_archive(f'/content/viettel-{PRECISION}-batch{BATCH_TOKENS}', 'zip', artifact_dir)
print('Download:', artifact)
